# Video Noise Analysis

Exploratory notebook.  
**Prerequisite:** `pip install -e .` from the repo root.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch

from videonoise.utils import get_device
from videonoise.io import load_video_folder, to_grayscale
from videonoise.metrics import (
    compute_all_metrics, frame_correlation,
    temporal_autocorrelation, temporal_ssim,
)
from videonoise.metrics.spectral import spatial_power_spectrum
from videonoise.noise import (
    simple_pixel_inversion, noise_statistics,
    cross_frame_correlation, NOISE_GENERATORS,
    noise_stats, temporal_correlation_noise,
)
from videonoise.analysis import (
    pixel_pca, plot_pca_maps, plot_pca_temporal,
    plot_acf_comparison, plot_power_spectra,
)

print('Device:', get_device())

Device: mps


## 1. Load videos

In [ ]:
RESIZE = (128, 128)   # (W, H)
MAX_FRAMES = 32

real_videos = load_video_folder('data/real/', max_frames=MAX_FRAMES, resize=RESIZE)
gen_videos  = load_video_folder('data/generated/', max_frames=MAX_FRAMES, resize=RESIZE)

print(f'Real: {len(real_videos)}   Generated: {len(gen_videos)}')
if real_videos:
    name, v = real_videos[0]
    print(f'Shape: {v.shape}  (T, C, H, W)')

## 2. Frame strip

In [ ]:
def show_frames(video, name, n=6):
    T = video.shape[0]
    idxs = np.linspace(0, T-1, n, dtype=int)
    fig, axes = plt.subplots(1, n, figsize=(3*n, 3))
    for ax, i in zip(axes, idxs):
        ax.imshow(video[i].permute(1, 2, 0).clamp(0, 1).numpy())
        ax.set_title(f't={i}')
        ax.axis('off')
    plt.suptitle(name); plt.tight_layout(); plt.show()

for name, v in real_videos[:2]: show_frames(v, f'REAL: {name}')
for name, v in gen_videos[:2]:  show_frames(v, f'GEN:  {name}')

## 3. Temporal ACF — real vs generated

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
lags = list(range(1, 11))

for name, v in real_videos[:4]:
    acf = temporal_autocorrelation(v)
    ax.plot(lags, [acf.get(f'lag_{k}', 0) for k in lags],
            'b-o', alpha=0.5, markersize=4)

for name, v in gen_videos[:4]:
    acf = temporal_autocorrelation(v)
    ax.plot(lags, [acf.get(f'lag_{k}', 0) for k in lags],
            'r-s', alpha=0.5, markersize=4)

from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([0],[0],color='b',label='Real'),
                   Line2D([0],[0],color='r',label='Generated')])
ax.axhline(0, color='gray', linestyle='--')
ax.set_xlabel('Lag (frames)'); ax.set_ylabel('ACF')
ax.set_title('Temporal ACF')
plt.tight_layout(); plt.show()

## 4. Spatial power spectrum

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for name, v in real_videos[:4]:
    ax.semilogy(spatial_power_spectrum(v)['radial_profile'], 'b-', alpha=0.4)
for name, v in gen_videos[:4]:
    ax.semilogy(spatial_power_spectrum(v)['radial_profile'], 'r-', alpha=0.4)
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([0],[0],color='b',label='Real'),
                   Line2D([0],[0],color='r',label='Generated')])
ax.set_xlabel('Spatial frequency'); ax.set_ylabel('Power')
ax.set_title('Radially-averaged 2D power spectrum')
plt.tight_layout(); plt.show()

## 5. Noise inversion

In [ ]:
for label, vids in [('Real', real_videos[:2]), ('Generated', gen_videos[:2])]:
    print(f'=== {label} ===')
    for name, v in vids:
        eps   = simple_pixel_inversion(v)
        s     = noise_statistics(eps)
        c     = cross_frame_correlation(eps)
        print(f'  {name}: mean={s["mean"]:.3f}  std={s["std"]:.3f}  '
              f'KL={s["kl_from_gaussian"]:.3f}  '
              f'cross-frame-corr={c["cross_frame_corr_mean"]:.3f}  '
              f'Gaussian(KS)={s["is_gaussian_ks"]}')

## 6. Noise init — AR(1) α sweep

In [ ]:
shape  = (16, 4, 32, 32)
alphas = [0.0, 0.3, 0.6, 0.8, 0.95]

fig, axes = plt.subplots(1, len(alphas), figsize=(4*len(alphas), 4))
for ax, alpha in zip(axes, alphas):
    eps   = NOISE_GENERATORS['ar1'](shape, alpha=alpha)
    corrs = temporal_correlation_noise(eps)
    ax.imshow(eps[0, 0].numpy(), cmap='RdBu_r', vmin=-3, vmax=3)
    ax.set_title(f'α={alpha}\ncorr={np.mean(corrs):.3f}')
    ax.axis('off')
plt.suptitle('AR(1) noise — temporal correlation α')
plt.tight_layout(); plt.show()

## 7. All noise types — spatial maps + histograms

In [ ]:
types  = list(NOISE_GENERATORS.keys())
kwargs = {'alpha': 0.8, 'sigma': 3.0, 'octaves': 4}

fig, axes = plt.subplots(2, len(types), figsize=(4*len(types), 8))
for col, nt in enumerate(types):
    eps = NOISE_GENERATORS[nt](shape, **kwargs)
    axes[0, col].imshow(eps[0, 0].numpy(), cmap='RdBu_r', vmin=-3, vmax=3)
    axes[0, col].set_title(nt); axes[0, col].axis('off')
    axes[1, col].hist(eps.flatten().numpy(), bins=60, density=True, alpha=0.7)
    x = np.linspace(-4, 4, 200)
    axes[1, col].plot(x, np.exp(-x**2/2)/np.sqrt(2*np.pi), 'r--')
    axes[1, col].set_xlim(-4, 4)
plt.suptitle('Noise types: spatial map (top) / histogram (bottom)')
plt.tight_layout(); plt.show()

## 8. Temporal consistency score

In [ ]:
for label, vids in [('Real', real_videos), ('Generated', gen_videos)]:
    scores = [temporal_ssim(v)['temporal_consistency_score'] for _, v in vids]
    if scores:
        print(f'{label}: TCS = {np.mean(scores):.4f} ± {np.std(scores):.4f}  (n={len(scores)})')

## 9. PCA spatial modes

In [ ]:
for name, v in (real_videos[:1] or gen_videos[:1]):
    res = pixel_pca(v, n_components=8)
    print(f'{name} — explained variance: {[f"{x:.1%}" for x in res["explained_variance_ratio"]]}')
    plot_pca_maps(res, 'results/plots/', name)
    plot_pca_temporal(res, 'results/plots/', name)